# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and Python data science tools.

### Dataset Source
The dataset is described via a [Croissant schema](https://mlcommons.org/croissant/) available at the URL below.

In [ ]:
# Install the mlcroissant library if needed
!pip install mlcroissant

## 1. Data Loading
We load the dataset metadata and available records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print a summary of the dataset
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's review the available record sets (`cr:RecordSet`) and their IDs, along with the fields in each record set.

In [ ]:
record_sets = dataset.record_sets
print(f"Total record sets: {len(record_sets)}\n")
for idx, rs in enumerate(record_sets):
    print(f"{idx+1}. RecordSet @id: {rs['@id']}")
    print(f"   name: {rs.get('name', 'N/A')}")
    # List field IDs in this RecordSet
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"   Field @ids:")
    for field in fields:
        if isinstance(field, dict):
            print(f"    - {field.get('@id')}")
        else:
            print(f"    - {field}")
    print("---------------------")

## 3. Data Extraction
Load data from specific record set(s) into DataFrame(s) for analysis.

**Note:** Use the `@id`s of RecordSets as seen in the overview above.

In [ ]:
# List all record set @id values for extraction
record_set_ids = [rs["@id"] for rs in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame with shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for: {record_set_id}")
    print("-----------------------------------")
# For illustration, select the first RecordSet for analysis
if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    example_df = dataframes[example_record_set_id]
    print(f"Using RecordSet {example_record_set_id} for further EDA.")
else:
    example_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Below we demonstrate filtering, normalization, and grouping using `@id` references to columns.

In [ ]:
if example_record_set_id is not None:
    df = example_df.copy()
    print("Columns in DataFrame:", df.columns.tolist())
    # Choose a numeric field via @id (try to guess numeric field for demo)
    # We'll attempt to select a likely numeric column: 'coverage', 'MW', 'peptides', etc.
    import re
    numeric_candidates = [col for col in df.columns if re.search(r'coverage|abundance|MW|peptides|count|score|intensity', col, re.I)]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Selected numeric field for analysis: {numeric_field_id}")

        # Ensure numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean()  # Use mean as demo threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a likely categorical field:
        possible_group_fields = [col for col in df.columns if re.search(r'accession|description|organism|sample|treatment|category|class|group', col, re.I)]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
                print(f"Grouped data by {group_field} (mean {numeric_field_id}):")
                print(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA demo.")
else:
    print("No record set DataFrame available for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields. Here, we create a histogram and boxplot for the chosen numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id is not None and numeric_candidates:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[numeric_field_id])
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.show()

    # Scatter plot with a possible group field if available
    if possible_group_fields:
        group_field = possible_group_fields[0]
        if df[group_field].nunique() < 20:
            plt.figure(figsize=(10,5))
            sns.boxplot(x=df[group_field], y=df[numeric_field_id])
            plt.title(f"{numeric_field_id} by {group_field}")
            plt.xticks(rotation=45)
            plt.show()
else:
    print("No record set DataFrame available for visualization.")

## 6. Conclusion

We loaded the FAIR<sup>2</sup> mass spectrometry dataset, explored available record sets and fields, and demonstrated basic data extraction and visualization. Further domain-specific analysis should reference the [dataset guide](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and use the `@id` fields for reproducible, schema-compliant workflows.